# От простого правила перцептрона к обучаемой нейронной сети

В этом уроке мы постепенно построим понятную картину обучения: сначала вручную изменим параметры простого классификатора, затем разберём объект `Perceptron`, а после перейдём к дифференцируемому нейрону, градиентам, потерям, оптимизаторам и инициализации. В конце мы используем библиотеку `open_neuron` и обучим многослойную сеть решать XOR.

После урока вы сможете объяснить:

- что вычисляют взвешенная сумма `z` и функция активации `a`;
- почему правило перцептрона работает для AND и OR, но не для XOR;
- чем ошибка предсказания отличается от функции потерь и её градиента;
- как цепное правило даёт градиенты весов и смещения;
- почему пороговый выход нельзя обучать обычным градиентным спуском;
- как `Neuron`, `Unit`, `Layer` и `NeuralNetwork` отличаются друг от друга.

Это учебное объяснение обычных полносвязных сетей. Оно не моделирует биологический мозг и не претендует на воспроизведение всех вариантов исторического перцептрона.


## 0. Термины и план

**Пример** — одна строка входных признаков `x`. **Выборка** — множество таких строк, записанное матрицей `X`. **Целевой ответ** обозначим `y`: это правильный класс или числовое значение для примера.

Модель с текущими параметрами вычисляет предсказание `a`. Во время обучения параметры меняются так, чтобы предсказания на обучающих примерах становились лучше. В коде ниже слова _ошибка_, _остаток_, _потеря_ и _градиент_ не взаимозаменяемы: мы отдельно разберём значение каждого.

Сначала будет простой **перцептронный алгоритм коррекции ошибки**. Он полезен для понимания классификатора, но это не общий алгоритм обучения любой нейронной сети и не backpropagation. Затем перейдём к гладкой функции активации и градиентному обучению.


Для векторных примеров дальше используется NumPy. Импортируем его до первого кода, который создаёт массивы.


In [ ]:
import numpy as np

## 1. Перцептрон в историческом контексте

Слово _перцептрон_ употреблялось для нескольких близких, но не одинаковых моделей. Статья Википедии, с которой начинался исходный конспект, приводит собственную историческую классификацию:

1. **Классический перцептрон Розенблатта** описывался слоями сенсорных `S`-, ассоциативных `A`- и реагирующих `R`-элементов.
2. **Однослойный перцептрон** в современном учебном употреблении — классификатор без скрытого слоя: входы напрямую влияют на выходы. В простейшем варианте его граница решений линейна.
3. **Многослойные модели Розенблатта** добавляли слои ассоциативных элементов, но устройство обучения могло отличаться от современного backpropagation.
4. **Многослойный перцептрон в популярном современном смысле** — сеть с обучаемыми скрытыми слоями; обучение таких сетей часто объясняют через backpropagation.

Это полезная карта терминов, а не универсальная классификация, которую все авторы используют одинаково. Когда далее мы говорим «перцептрон», мы имеем в виду простой пороговый линейный классификатор и правило обновления его весов.

Розенблатт предложил перцептрон в конце 1950-х. В демонстрации 1958 года использовалась симуляция на IBM 704; её следует отличать от специализированного аппаратного устройства Mark I. У Mark I была более сложная схема с `S`-, `A`- и `R`-элементами. Позднее книга Минского и Пейперта _Perceptrons_ подробно разобрала ограничения определённых классов моделей. Влияние этой книги на ход исследований важно, но историю последующего спада нельзя сводить к одной публикации или одной причине.

В 1986 году Rumelhart, Hinton и Williams опубликовали влиятельное описание обучения сетей со скрытыми единицами методом распространения градиента. Это важный этап популяризации backpropagation для нейронных сетей, но не основание приписывать им единоличное изобретение всех идей обратного распространения.

Источники и более точные исторические оговорки собраны в конце урока.


## 2. Самая простая модель: один вход, один вес

Начнём с одного числового входа `x`, одного веса `w` и пороговой активации. Сначала модель вычисляет

$$
z = xw.
$$

Затем превращает число в класс:

$$
a = \begin{cases}1,&z>0,\\0,&z\le 0.\end{cases}
$$

Число `z` до порога называют **взвешенной суммой** или **значением до активации**. `a` — выход после активации. Для начала зададим `x = 1`, правильный ответ `y = 1`, а вес возьмём отрицательным.


In [ ]:
x = 1.0
expected = 1
weight = -2.75

for epoch in range(5):
  z = x * weight
  prediction = int(z >= 0)
  print(
      f"Эпоха {epoch + 1}: w={weight:.2f}, z={z:.2f}, "
      f"предсказание={prediction}, цель={expected}"
  )

  if prediction != expected:
    weight += x if expected == 1 else -x

Эпоха 1: w=-2.75, z=-2.75, предсказание=0, цель=1
Эпоха 2: w=-1.75, z=-1.75, предсказание=0, цель=1
Эпоха 3: w=-0.75, z=-0.75, предсказание=0, цель=1
Эпоха 4: w=0.25, z=0.25, предсказание=1, цель=1
Эпоха 5: w=0.25, z=0.25, предсказание=1, цель=1


В этой маленькой задаче при каждом неправильном ответе мы двигаем вес в нужную сторону. После того как `z` пересекает ноль, предсказание становится правильным и вес перестаёт меняться. Это наглядная иллюстрация обучения, но правило обновления пока подобрано специально для порогового классификатора.

В общем случае удобнее записать его так. Для меток `0` и `1` определим остаток `e = y - a`, а затем обновим параметры:

$$
w_i \leftarrow w_i + \eta e x_i, \qquad b \leftarrow b + \eta e.
$$

`η` — learning rate, или скорость обучения. Формула задаёт шаг `η` умножить на соответствующий градиентоподобный сигнал; она **не означает**, что параметры меняются на определённый процент. В простом правиле перцептрона `η` часто равен `1`.


### 2.1 Добавляем смещение

Смещение `b` сдвигает порог независимо от входных признаков:

$$
z = xw + b.
$$

Даже при `x = 0` значение `z` может отличаться от нуля. Если правильная метка равна нулю, а модель ошибочно выдаёт единицу, тот же остаток `e = y-a` уменьшает и вес, и смещение.


In [ ]:
x = 1.0
expected = 0
weight = -2.75
bias = 5.25
threshold = 0.0

for epoch in range(5):
  z = x * weight + bias
  prediction = int(z > threshold)
  error = expected - prediction
  print(
      f"Эпоха {epoch + 1}: w={weight:.2f}, b={bias:.2f}, "
      f"z={z:.2f}, предсказание={prediction}, остаток={error}"
  )
  weight += error * x
  bias += error

Эпоха 1: w=-2.75, b=5.25, z=2.50, предсказание=1, остаток=-1
Эпоха 2: w=-3.75, b=4.25, z=0.50, предсказание=1, остаток=-1
Эпоха 3: w=-4.75, b=3.25, z=-1.50, предсказание=0, остаток=0
Эпоха 4: w=-4.75, b=3.25, z=-1.50, предсказание=0, остаток=0
Эпоха 5: w=-4.75, b=3.25, z=-1.50, предсказание=0, остаток=0


### 2.2 Несколько признаков и примеров

У примера могут быть несколько признаков, например `x = [x₁, x₂]`. Тогда

$$
z = x_1w_1 + x_2w_2 + b = \mathbf{x}\cdot\mathbf{w}+b.
$$

Векторное обновление перцептрона имеет вид `w += η * error * x`. Если признак равен нулю, его вес в этом конкретном обновлении не меняется, поскольку вклад `error * xᵢ` равен нулю. Это не означает, что такой вес никогда не обучится: другие примеры могут иметь ненулевое значение этого признака.

Ниже сравним два объекта. Они похожи на часть таблицы AND: `[1, 0]` должен относиться к классу `0`, `[1, 1]` — к классу `1`.


In [ ]:
X_pair = np.array([[1.0, 0.0], [1.0, 1.0]])
y_pair = np.array([0, 1])
weights = np.array([-0.5, -3.25])
bias = 3.25

for epoch in range(8):
  for x, target in zip(X_pair, y_pair):
    z = x @ weights + bias
    prediction = int(z >= 0.5)
    error = target - prediction
    weights += error * x
    bias += error

  predictions = (X_pair @ weights + bias >= 0.5).astype(int)
  print(
      f"Эпоха {epoch + 1}: веса={weights}, bias={bias:.2f}, ответы={predictions}")

Эпоха 1: веса=[-0.5  -2.25], bias=3.25, ответы=[1 1]
Эпоха 2: веса=[-0.5  -1.25], bias=3.25, ответы=[1 1]
Эпоха 3: веса=[-0.5  -0.25], bias=3.25, ответы=[1 1]
Эпоха 4: веса=[-1.5  -0.25], bias=2.25, ответы=[1 1]
Эпоха 5: веса=[-1.5   0.75], bias=2.25, ответы=[1 1]
Эпоха 6: веса=[-1.5   1.75], bias=2.25, ответы=[1 1]
Эпоха 7: веса=[-2.5   1.75], bias=1.25, ответы=[0 1]
Эпоха 8: веса=[-2.5   1.75], bias=1.25, ответы=[0 1]


Этот цикл — **онлайн-обучение**: параметры обновляются после каждого примера. Порядок примеров может влиять на промежуточные веса и число эпох. В более поздних примерах мы будем отдельно говорить об обновлении после одного примера, пакета примеров или всей выборки.

Обратите внимание: в этом коде порог `0.5` применяется к `z`, а не к вероятности. Это просто выбранная граница для данного классификатора. Утверждать, что это «середина между классами 0 и 1», нельзя: `z` — не класс и не вероятность.


## 3. Класс `Perceptron`: состояние и операции

Теперь соберём эту маленькую модель в объект. **Класс** описывает общую структуру, а **объект** хранит конкретные веса, смещение и последнее вычисленное состояние.

- **Атрибуты** хранят состояние: `weights`, `bias`, `inputs`, `z`, `a`.
- **Методы** выполняют операции над этим состоянием: `forward`, `update`, `fit`, `predict`.
- `__init__` вызывается при создании объекта; `__call__` позволяет вызвать объект как функцию; `__str__` задаёт его печатное представление.

Обычный вызов `perceptron.predict(X)` часто яснее, чем `perceptron(X)`. `__call__` показан здесь для изучения Python, а не потому, что он обязателен для нейронной сети.


In [ ]:
class Perceptron:
  def __init__(self, weights, bias=0.0, threshold=0.5):
    self.weights = np.asarray(weights, dtype=float).copy()
    self.bias = float(bias)
    self.threshold = float(threshold)
    self.inputs = None
    self.z = None
    self.a = None

  def forward(self, inputs):
    inputs = np.asarray(inputs, dtype=float)
    if inputs.shape != self.weights.shape:
      raise ValueError(
          "Число входных признаков должно совпадать с числом весов")
    self.inputs = inputs
    self.z = float(inputs @ self.weights + self.bias)
    self.a = int(self.z >= self.threshold)
    return self.a

  def update(self, target, learning_rate=1.0):
    error = target - self.a
    self.weights += learning_rate * error * self.inputs
    self.bias += learning_rate * error
    return error

  def fit(self, X, y, epochs=10, learning_rate=1.0):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int).reshape(-1)
    if X.ndim != 2 or X.shape[0] != y.shape[0]:
      raise ValueError(
          "X должен быть матрицей с одной меткой на каждую строку")

    mistake_history = []
    for _ in range(epochs):
      mistakes = 0
      for inputs, target in zip(X, y):
        prediction = self.forward(inputs)
        if prediction != target:
          self.update(target, learning_rate)
          mistakes += 1
      mistake_history.append(mistakes)
      if mistakes == 0:
        break
    return mistake_history

  def predict(self, X):
    return np.array([self.forward(row) for row in np.asarray(X)])

  def __call__(self, X):
    return self.predict(X)

  def __str__(self):
    return (
        f"Perceptron(weights={self.weights}, bias={self.bias:.3f}, "
        f"threshold={self.threshold:.3f}, z={self.z}, output={self.a})"
    )

`fit` здесь делает полный проход по обучающим примерам, обновляя параметры после каждого неправильного ответа. Он останавливается раньше заданного числа эпох, если на всём наборе нет ошибок. Вызов `forward` сохраняет последнее состояние, поэтому строка `print(model)` показывает последний обработанный пример.

Такое объектное устройство удобно для учебной модели с собственными параметрами. Но из этого не следует, что библиотека обязана хранить каждый нейрон отдельным вычислительным объектом: матричные вычисления слоя обычно выполняются над всей группой нейронов сразу.


## 4. AND, OR и линейная разделимость

Для двух бинарных входов у каждой логической операции четыре примера:

| `x₁` | `x₂` | AND |  OR | XOR |
| ---: | ---: | --: | --: | --: |
|    0 |    0 |   0 |   0 |   0 |
|    0 |    1 |   0 |   1 |   1 |
|    1 |    0 |   0 |   1 |   1 |
|    1 |    1 |   1 |   1 |   0 |

У AND и OR положительные точки можно отделить от отрицательных одной прямой. Поэтому пороговый линейный классификатор может выразить эти функции. Один такой классификатор задаёт границу `x·w + b = threshold`.


In [ ]:
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])

for name, targets in (("AND", y_and), ("OR", y_or)):
  model = Perceptron(weights=[-0.5, -1.25], bias=1.75)
  mistakes = model.fit(X_logic, targets, epochs=20)
  predictions = model.predict(X_logic)
  print(f"{name}: предсказания={predictions}, ожидается={targets}")
  print(
      f"  ошибки по эпохам={mistakes}; веса={model.weights}, bias={model.bias:.2f}")

AND: предсказания=[0 0 0 1], ожидается=[0 0 0 1]
  ошибки по эпохам=[2, 3, 3, 2, 3, 2, 1, 0]; веса=[2.5  0.75], bias=-2.25
OR: предсказания=[0 1 1 1], ожидается=[0 1 1 1]
  ошибки по эпохам=[2, 2, 1, 2, 2, 1, 0]; веса=[1.5  0.75], bias=-0.25


В таблице XOR положительные классы находятся по диагонали: `[0, 1]` и `[1, 0]`. Одной прямой их отделить от `[0, 0]` и `[1, 1]` нельзя.

Это можно увидеть и без графика. Перенесём порог в смещение и обозначим `c = b - threshold`, так что выход равен `1`, если `x₁w₁ + x₂w₂ + c ≥ 0`. Для правильного XOR:

- пример `[0, 0] → 0` требует `c < 0`;
- пример `[1, 0] → 1` требует `w₁ + c ≥ 0`;
- пример `[0, 1] → 1` требует `w₂ + c ≥ 0`.

Сложив последние два неравенства, получим `w₁ + w₂ + 2c ≥ 0`. Так как `c < 0`, отсюда следует `w₁ + w₂ + c ≥ -c > 0`. Но для `[1, 1] → 0` нужно `w₁ + w₂ + c < 0`. Получилось противоречие. Значит, для стандартного однослойного порогового классификатора решения нет — больше эпох или другой learning rate этого ограничения выразимости не устранят.


In [10]:
xor_perceptron = Perceptron(weights=[-0.5, -1.25], bias=1.75)
xor_mistakes = xor_perceptron.fit(X_logic, np.array([0, 1, 1, 0]), epochs=20)
xor_predictions = xor_perceptron.predict(X_logic)
print("XOR: предсказания =", xor_predictions)
print("XOR: правильные    =", np.array([0, 1, 1, 0]))
print("Число ошибок по эпохам:", xor_mistakes)

XOR: предсказания = [1 0 0 0]
XOR: правильные    = [0 1 1 0]
Число ошибок по эпохам: [3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


Эксперимент показывает не «плохую настройку», а ограничение **этого класса моделей**. Скрытый слой с нелинейной активацией создаёт промежуточные признаки, в пространстве которых выходной слой может отделить XOR. Ниже мы вернёмся к этому примеру уже с `NeuralNetwork`.


## 5. От порогового перцептрона к дифференцируемому нейрону

В библиотеке есть два понятия, которые важно не смешивать:

- самостоятельный `Neuron` хранит свои веса и смещение, считает прямой проход, градиенты и может обучаться отдельно;
- `Unit` внутри `Layer` предоставляет доступ к параметрам и состоянию одного нейрона, принадлежащим матрицам слоя. Сам `Unit` не обучается отдельно.

Абстрактный нейрон с различными функциями активации шире порогового перцептрона. В этом уроке мы используем sigmoid, потому что её выход плавно меняется и у неё есть удобная производная:

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad \sigma'(z)=\sigma(z)(1-\sigma(z)).
$$

Плавная активация позволяет вычислить, как небольшое изменение веса повлияет на выход. Ступенчатая Heaviside почти всюду имеет нулевую производную и в точке скачка не дифференцируема. Поэтому нельзя выдавать массив единиц за её настоящую производную. Если специально применяют surrogate-gradient, это отдельное приближение, а не производная Heaviside.


In [11]:
from open_neuron.activations import heaviside, linear, relu, sigmoid, softplus, tanh

values = np.array([-2.0, 0.0, 2.0])
print("z:                  ", values)
print("sigmoid(z):         ", sigmoid(values))
print("sigmoid'(z):        ", sigmoid(values, derivative=True))
print("tanh(z):            ", tanh(values))
print("ReLU(z):            ", relu(values))
print("Softplus(z):        ", softplus(values))
print("linear(z):          ", linear(values))
print("Heaviside(z):       ", heaviside(values))
print("Heaviside'(z) API:  ", heaviside(values, derivative=True))

z:                   [-2.  0.  2.]
sigmoid(z):          [0.11920292 0.5        0.88079708]
sigmoid'(z):         [0.10499359 0.25       0.10499359]
tanh(z):             [-0.96402758  0.          0.96402758]
ReLU(z):             [0. 0. 2.]
Softplus(z):         [0.12692801 0.69314718 2.12692801]
linear(z):           [-2.  0.  2.]
Heaviside(z):        [0. 0. 1.]
Heaviside'(z) API:   [0. 0. 0.]


Последняя строка показывает соглашение конкретного API: библиотека возвращает нулевой массив как выбранное значение производной Heaviside. В точке скачка настоящая производная не определена, а вне скачка производная равна нулю. Следовательно, такую активацию нельзя обучать обычным backprop с полезным градиентом; для ручного перцептронного примера выше мы использовали отдельное правило коррекции ошибки.

NumPy позволяет передавать в эти функции сразу вектор или матрицу. Форма результата сохраняется: например, пакет входов имеет форму `(batch_size, input_size)`, а выход одного нейрона — `(batch_size, 1)`.


## 6. Прямой проход и функции потерь

Для одного нейрона с несколькими входами:

$$
\mathbf{z}=XW+b, \qquad A=\sigma(\mathbf{z}).
$$

Если `X` имеет форму `(N, F)`, где `N` — число примеров, а `F` — число признаков, то веса одного выходного нейрона имеют форму `(F, 1)`, смещение — `(1, 1)`, а `Z` и `A` — `(N, 1)`.

**Остаток** `A - Y` сообщает знак и величину расхождения для каждого элемента. **Функция потерь** сводит эти расхождения к числу, например среднеквадратичная ошибка:

$$
L_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(a_i-y_i)^2.
$$

Остаток сам по себе ещё не градиент параметров. Чтобы узнать, как менять `W` и `b`, нужно вычислить производные потерь и активации.


In [12]:
from open_neuron.losses import CE, MAE, MSE
from open_neuron.neurons import Neuron
from open_neuron.optimizers import Adam, SGD

X_and = X_logic
Y_and = y_and.reshape(-1, 1).astype(float)

neuron = Neuron(
    weights=[0.45, -0.12],
    bias=1.0,
    input_size=2,
    activation=sigmoid,
    loss=MSE(),
    optimizer=SGD(learning_rate=0.7, momentum=0.3),
)

A_initial = neuron.forward(X_and)
print("Форма X, W, b, Z, A:", X_and.shape, neuron.weights.shape, neuron.bias.shape,
      neuron.Z.shape, A_initial.shape)
print("Начальные Z:       ", neuron.Z.ravel())
print("Начальные A:       ", A_initial.ravel())
print("Целевые значения Y:", Y_and.ravel())
print("Начальная MSE:     ", neuron.loss_function.evaluate_loss(Y_and, A_initial))

Форма X, W, b, Z, A: (4, 2) (2, 1) (1, 1) (4, 1) (4, 1)
Начальные Z:        [1.   0.88 1.45 1.33]
Начальные A:        [0.73105858 0.70682222 0.80999843 0.79084063]
Целевые значения Y: [0. 0. 0. 1.]
Начальная MSE:      0.4334723501835361


## 7. Backpropagation для одного нейрона

Backpropagation — это применение цепного правила к последовательности вычислений сети. Даже у одного нейрона можно явно увидеть цепочку:

$$
W,b \longrightarrow Z=XW+b \longrightarrow A=\sigma(Z) \longrightarrow L(A,Y).
$$

Сначала градиент потерь по выходу:

$$
\frac{\partial L}{\partial A_i}=\frac{2(A_i-y_i)}{N}.
$$

Затем переносим его через sigmoid:

$$
\delta_i=\frac{\partial L}{\partial Z_i}
=\frac{\partial L}{\partial A_i}\sigma'(Z_i).
$$

Наконец, поскольку `Z = XW+b`:

$$
\frac{\partial L}{\partial W}=X^T\delta,\qquad
\frac{\partial L}{\partial b}=\sum_{i=1}^{N}\delta_i.
$$

`δ` здесь — градиент потерь по **предактивации `Z`**, а не по входам нейрона. Градиент по входам был бы `δ @ W.T`; он нужен, когда нужно передать влияние ошибки дальше к предыдущему слою. Самостоятельный `Neuron` не обязан возвращать такой градиент для обучения предшествующего слоя.


In [13]:
error = neuron.loss_function.evaluate_error(Y_and, A_initial)
neuron.backward(error)
print("Остатки A - Y:    ", error.ravel())
print("Градиент dL/dW:   ", neuron.gradient["weights"].ravel())
print("Градиент dL/db:   ", neuron.gradient["bias"].ravel())

Остатки A - Y:     [ 0.73105858  0.70682222  0.80999843 -0.20915937]
Градиент dL/dW:    [0.04503107 0.05593676]
Градиент dL/db:    [0.19013395]


В `MSE.loss_derivative` библиотеки деление выполняется на число элементов матрицы остатка. Поэтому усреднение согласовано и с числом примеров, и с числом выходов. Для одного выхода на пример знаменатель равен `N`; для нескольких — произведению числа примеров на число выходов.

Посмотрим на один шаг оптимизатора. Для vanilla SGD параметр обновляется как `θ ← θ - η ∂L/∂θ`. Знак минус означает движение против градиента потерь.


In [14]:
loss_before = neuron.loss_function.evaluate_loss(Y_and, neuron.predict(X_and))
weights_before = neuron.weights.copy()
bias_before = neuron.bias.copy()
neuron.update()
loss_after = neuron.loss_function.evaluate_loss(Y_and, neuron.predict(X_and))
print("Веса до шага:      ", weights_before.ravel())
print("Веса после шага:   ", neuron.weights.ravel())
print("Смещение до/после: ", bias_before.ravel(), neuron.bias.ravel())
print(f"MSE до/после:       {loss_before:.6f} -> {loss_after:.6f}")

Веса до шага:       [ 0.45 -0.12]
Веса после шага:    [ 0.41847825 -0.15915573]
Смещение до/после:  [1.] [0.86690623]
MSE до/после:       0.433472 -> 0.404537


Убывание потерь после одного шага зависит от данных и величины шага; слишком большой learning rate может увеличить потери. Сам градиент говорит о локальном направлении роста функции, но не гарантирует, что любой выбранный шаг окажется удачным.

**MAE** вычисляет среднее абсолютное отклонение; её производная по выходу — `sign(A-Y)/M`, где `M` — число элементов. В нуле выбирают субградиент, например `0`. **MSE** сильнее штрафует большие отклонения и имеет градиент `2(A-Y)/M`.

**Categorical cross-entropy** используется с распределением вероятностей softmax и one-hot метками:

$$
L=-\frac{1}{N}\sum_i\sum_c y_{ic}\log(p_{ic}).
$$

Для совместного выхода softmax и этой функции потерь градиент по логитам упрощается до `(P-Y)/N`. Это fused-градиент для указанной пары, а не производная кросс-энтропии по вероятностям и не общая производная softmax.


In [ ]:
probabilities = np.array([[0.8, 0.2], [0.1, 0.9]])
one_hot = np.array([[1.0, 0.0], [0.0, 1.0]])
cross_entropy = CE()
ce_error = cross_entropy.evaluate_error(one_hot, probabilities)
print("MSE примера:", MSE().evaluate_loss(
    np.array([[1.0], [0.0]]), np.array([[0.8], [0.1]])))
print("MAE примера:", MAE().evaluate_loss(
    np.array([[1.0], [0.0]]), np.array([[0.8], [0.1]])))
print("CCE примера:", cross_entropy.evaluate_loss(one_hot, probabilities))
print("Fused dL/dZ:", cross_entropy.loss_derivative(ce_error))

MSE примера: 0.02499999999999999
MAE примера: 0.14999999999999997
CCE примера: 0.164252033486018
Fused dL/dZ: [[-0.1   0.1 ]
 [ 0.05 -0.05]]


## 8. Обучение `Neuron` и вызов объекта

Метод `fit` объединяет уже разобранные операции в цикл: получить пакет, выполнить forward, вычислить остаток, выполнить backward и обновить параметры. В `predict` параметры не меняются; считается только прямой проход.

У этого класса `Neuron` собственные веса и смещение. В сети же `Unit` смотрит на один нейрон матричного `Layer`, а обучение и обновления выполняются слоями и `NeuralNetwork`. Это не два названия одного класса, а две разные роли.


In [16]:
trained_neuron = Neuron(
    weights=[0.45, -0.12],
    bias=1.0,
    input_size=2,
    activation=sigmoid,
    loss=MSE(),
    optimizer=SGD(learning_rate=0.7, momentum=0.3),
)
trained_neuron.fit(X_and, Y_and, epochs=100, batch_size=1, shuffle=False)
probabilities_and = trained_neuron.predict(X_and)
classes_and = (probabilities_and >= 0.5).astype(int)
print("Вероятности AND:", probabilities_and.ravel())
print("Классы:          ", classes_and.ravel())
print("Ожидалось:       ", Y_and.astype(int).ravel())
print("Последнее состояние:", trained_neuron)

Training started with Overall Loss 0.4335 on Train Data
Epoch 100/100, Loss: 0.0154, Validation Loss: 0.0145 on Train Data
Training completed with Overall Loss 0.0145
Вероятности AND: [0.00412337 0.1311449  0.13379389 0.84918921]
Классы:           [0 0 0 1]
Ожидалось:        [0 0 0 1]
Последнее состояние: Neuron 2, inputs: [1. 1.], weights: [3.6191 3.5961], bias: [-5.487], z: 1.7282560416241477, activation: sigmoid, a: 0.8491892129058849


Sigmoid возвращает непрерывные значения от нуля до единицы. Их можно трактовать как оценки уверенности/вероятности в бинарной задаче, но сами по себе числа ещё не являются классами. Для простого вывода класса применяем правило `класс 1, если sigmoid(z) >= 0.5`, иначе `класс 0`. Это не тот же самый порог, который раньше применялся к `z` в пороговом перцептроне.

Скрытое состояние объекта сохраняет последний обработанный пакет. Поэтому после `predict(X)` строковое представление показывает состояние последней строки `X` — это отладочный вывод, а не дополнительное предсказание.


## 9. Пакеты данных, эпохи и обобщение

- **Одна итерация обновления** использует один пример либо один пакет — это зависит от `batch_size`.
- **Эпоха** означает один проход по всем обучающим примерам.
- **Batch Gradient Descent** обновляет параметры после всей обучающей выборки.
- **Stochastic Gradient Descent** обновляет параметры после отдельного примера; в точной терминологии пример обычно выбирается случайно, хотя можно пройти выборку в перемешанном порядке.
- **Mini-batch Gradient Descent** обновляет параметры после небольшого пакета.

В API `fit` можно менять `batch_size`, включать перемешивание и передавать `validation_data`. AND/OR/XOR-таблицы здесь слишком малы и перечисляют весь конечный набор входов, поэтому мы учимся на всех четырёх комбинациях и проверяем на них же. Для реальных данных оценивать качество только на обучающей выборке недостаточно: отложенную проверочную выборку отделяют заранее и не используют для обновления весов.


## 10. Оптимизаторы

Оптимизатор преобразует градиент в изменение параметров. Самый простой шаг градиентного спуска уже является оптимизацией:

$$
\theta_{t+1}=\theta_t-\eta g_t.
$$

Поэтому базовый вариант, который только вычитает learning rate, умноженный на градиент, не «ничего не оптимизирует» — это vanilla SGD. В библиотеке `SGD` дополнительно поддерживает momentum: он смешивает текущий градиент с предыдущим изменением.

$$
u_t=\eta g_t+\mu u_{t-1},\qquad \theta_{t+1}=\theta_t-u_t.
$$

RMSprop нормирует градиент по скользящему среднему квадратов градиентов. Adam объединяет скользящие средние самих градиентов и их квадратов, корректирует начальное смещение этих средних и использует адаптивный шаг для каждого параметра. В текущем API реализованы SGD с momentum и Adam; RMSprop здесь упомянут для сравнения, но отдельного класса оптимизатора в библиотеке нет. Эти алгоритмы стоит разбирать после того, как понятен обычный шаг по градиенту.

Термины Batch, Stochastic и Mini-batch описывают **сколько примеров участвует в одном обновлении**. Momentum, RMSprop и Adam описывают **как градиенты превращаются в изменение параметров**. Это две разные оси классификации.


In [17]:
sgd = SGD(learning_rate=0.1, momentum=0.0)
sgd_with_momentum = SGD(learning_rate=0.1, momentum=0.9)
adam = Adam(learning_rate=0.01)
print("Vanilla SGD:", sgd)
print("SGD с momentum:", sgd_with_momentum)
print("Adam:", adam)

Vanilla SGD: "SGD"
SGD с momentum: "SGD"
Adam: "Adam"


## 11. Инициализация весов

На маленьком примере мы могли выбрать веса вручную и проследить арифметику. В настоящем слое весов много, поэтому перед обучением их обычно инициализируют автоматически.

Случайные небольшие веса позволяют нейронам одного слоя начать с разными значениями. Если всем нейронам слоя задать одинаковые веса и одинаковое смещение, при одинаковом входе они будут давать одинаковый выход и получать одинаковые градиенты, поэтому останутся похожими друг на друга. Именно симметрию весов скрытых нейронов важно нарушить. Смещения при этом часто инициализируют нулями.

**Glorot/Xavier** масштабирует значения с учётом `fan_in` и `fan_out`; равномерный вариант использует границы `±sqrt(6/(fan_in+fan_out))`. **He** учитывает число входов и часто используется с ReLU; равномерный вариант использует `±sqrt(6/fan_in)`. Это распространённые начальные масштабы, а не гарантия, что обучение сойдётся или даст максимальную точность.


In [18]:
from open_neuron.initializers import Constant, GlorotUniform, HeNormal

np.random.seed(7)
print("Константа ноль, форма (2, 3):\n", Constant(0)((2, 3)))
print("Glorot Uniform, форма (2, 3):\n", GlorotUniform()((2, 3)))
print("He Normal, форма (2, 3):\n", HeNormal()((2, 3)))

Константа ноль, форма (2, 3):
 [[0 0 0]
 [0 0 0]]
Glorot Uniform, форма (2, 3):
 [[-0.92826203  0.61327135 -0.13493861]
 [ 0.48958767  1.04722255  0.08434023]]
He Normal, форма (2, 3):
 [[-7.88923029e-01  2.06557291e-03 -8.90385858e-04]
 [-1.75472431e+00  1.01765801e+00  6.00498516e-01]]


## 12. Почему скрытый слой позволяет решить XOR

Один пороговый перцептрон задаёт одну линейную границу. Многослойная сеть сначала применяет несколько преобразований и нелинейную активацию к скрытым нейронам, затем строит выход из их результатов. Эти промежуточные признаки могут сделать XOR разделимым для выходного слоя.

Теперь решим ту же задачу текущим API. Два sigmoid-нейрона образуют скрытый слой; выходной слой с двумя классами использует softmax. Для каждой строки XOR-таблицы целевая метка записана one-hot-вектором.


In [19]:
from open_neuron.activations import softmax
from open_neuron.layers import Layer
from open_neuron.models import NeuralNetwork

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0, 1], [1, 0], [1, 0], [0, 1]])
class_names = np.array(["входы различаются", "входы равны"])

np.random.seed(7)
xor_model = NeuralNetwork(
    [
        Layer(2, activation=sigmoid),
        Layer(2, activation=softmax),
    ],
    inputs_dim=X_xor.shape[1],
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.1),
)
xor_model.fit(X_xor, y_xor, epochs=500, batch_size=4, shuffle=False)
probabilities_xor = xor_model.predict(X_xor)
predicted_xor = np.argmax(probabilities_xor, axis=1)
expected_xor = np.argmax(y_xor, axis=1)
print("Вероятности [различаются, равны]:\n", probabilities_xor)
print("Предсказанные классы:", class_names[predicted_xor])
print("Ожидаемые классы:    ", class_names[expected_xor])

Training started with Overall Loss 0.6971 on Train Data
Epoch 500/500, Loss: 0.0012, Validation Loss: 0.0012 on Train Data
Training completed with Overall Loss 0.0012
Вероятности [различаются, равны]:
 [[0.00107084 0.99892916]
 [0.99891104 0.00108896]
 [0.99889685 0.00110315]
 [0.0016812  0.9983188 ]]
Предсказанные классы: ['входы равны' 'входы различаются' 'входы различаются' 'входы равны']
Ожидаемые классы:     ['входы равны' 'входы различаются' 'входы различаются' 'входы равны']


Скрытый слой не просто добавляет больше весов: благодаря нелинейной активации он позволяет строить нелинейные границы решений. Если бы каждый слой содержал только линейное преобразование, их последовательная композиция всё ещё была бы линейным преобразованием и XOR остался бы недоступен.

## 13. Что важно запомнить

1. Перцептронный алгоритм коррекции ошибки и градиентное обучение — родственные по цели, но разные правила.
2. Порог на предактивации `z` не является порогом вероятности на выходе sigmoid.
3. `error = A - Y` — остаток; `loss` сводит расхождения к числу; gradient показывает, как параметры влияют на loss.
4. Backpropagation применяет цепное правило: для одного нейрона `δ = dL/dA · da/dZ`, `dL/dW = Xᵀδ`, `dL/db = sum(δ)`.
5. Heaviside не имеет обычного полезного градиента для такого обучения. Для классического перцептрона используйте его правило коррекции; для backprop — дифференцируемую активацию.
6. Один линейный пороговый классификатор решает линейно разделимые задачи, например AND/OR, но не XOR. Нелинейный скрытый слой может представить XOR.
7. `Neuron` владеет собственными параметрами; `Unit` представляет нейрон внутри параметрических матриц `Layer`; сеть организует обучение слоёв.
8. SGD, momentum и Adam — способы обновлять параметры по градиентам; размер пакета — отдельная настройка цикла обучения.
9. Инициализация помогает слоям начать обучение с разумного масштаба и различающихся весов, но сама по себе не гарантирует качество.

### Термины

- **Эпоха:** проход по всем обучающим примерам.
- **Learning rate:** множитель шага обновления.
- **Logit:** значение до softmax.
- **Градиент:** производные функции потерь по параметрам или промежуточным значениям.
- **Линейная разделимость:** существование одной гиперплоскости, отделяющей классы.
- **Backpropagation:** вычисление градиентов по слоям сети с цепным правилом.

### Связанные примеры библиотеки

- [Обучение одиночного нейрона](neuron_learning.ipynb) — прямой проход, градиенты и обновление `Neuron`.
- [AND и OR](classification_01_and_or.ipynb) — обучение одного sigmoid-нейрона на всей таблице.
- [XOR](classification_02_xor.ipynb) — многослойная классификация через текущий API.

### Источники по истории

- [Историческая классификация перцептронов в Википедии](https://ru.wikipedia.org/wiki/Перцептрон#Историческая_классификация) — использована как один вариант терминологической карты, не как универсальный стандарт.
- [Cornell: демонстрация перцептрона на IBM 704 в 1958 году](https://news.cornell.edu/stories/2019/09/professors-perceptron-paved-way-ai-60-years-too-soon).
- [Smithsonian: устройство Mark I и его S-, A- и R-элементы](https://www.si.edu/object/electronic-neural-network-mark-i-perceptron:nmah_334414).
- [Olazaran: исторический анализ спора вокруг книги _Perceptrons_](https://journals.sagepub.com/doi/10.1177/030631296026003005).
- [Rumelhart, Hinton и Williams: _Learning representations by back-propagating errors_ (1986)](https://doi.org/10.1038/323533a0).
